In [1]:
import numpy as np
import torch
from torch.distributions.multivariate_normal import MultivariateNormal
from scipy.stats import chi2
import time
from tqdm import tqdm

class GaussianMixtureModel:
    def __init__(self, means: torch.Tensor, covariances: torch.Tensor, weights: torch.Tensor,
                 percentile=0.80, delta=0.05, inflate_scale=5.0, inflate_full=False, sub_dims=None, device='cpu'):
        self.device = device
        self.means = means
        self.covariances = covariances
        self.weights = weights

        # assert torch.isclose(self.weights.sum(), torch.tensor(1.0, device=self.device)), 'weights need to sum up to 1'
        # assert len(means) == len(covariances)
        # assert len(covariances) == self.weights.shape[0]

        self.num_cluster = len(means)

        d = self.means[0].shape[0]
        self.d = d
        n = d if inflate_full else np.random.randint(1, d + 1)  # Generate random integer between 1 and d, inclusive
        if inflate_full and sub_dims is not None:
            print('we are inflating all the dimensions, however, sub_dims is provided')
            raise Exception
        self.sub_dims = torch.sort(torch.randperm(d)[:n]).values.to(self.device) if sub_dims is None else sub_dims.to(
            self.device)

        self.threshold_plus_delta = chi2.ppf(percentile + delta, df=len(self.sub_dims))
        self.threshold_minus_delta = chi2.ppf(percentile - delta, df=len(self.sub_dims))

        self.inflated_covariances = []
        self.inv_sub_covariances = []

        for cov in self.covariances:
            cov_copy = cov.clone()
            sub_cov = cov_copy[self.sub_dims, :][:, self.sub_dims]  # the sub_cov defines a new (smaller) Gaussian
            # n x n ---> sub_dims x n ---> sub_dims x sub_dims
            self.inv_sub_covariances.append(torch.linalg.inv(sub_cov))
            cov_copy[self.sub_dims[:, None], self.sub_dims] *= inflate_scale
            self.inflated_covariances.append(cov_copy)

        self.inflated_covariances = torch.stack(self.inflated_covariances)
        self.inv_sub_covariances = torch.stack(self.inv_sub_covariances)

        self.GMM4sample = [MultivariateNormal(self.means[cluster_id], self.covariances[cluster_id])
                           for cluster_id in range(len(self.weights))]

        self.GMM4inf = [MultivariateNormal(self.means[cluster_id], self.inflated_covariances[cluster_id])
                        for cluster_id in range(len(self.weights))]
        
        target_dims = 100
        target_clusters = 5
        self.ret_means = self.means.contiguous()
        self.ret_variances = torch.diagonal(self.covariances, dim1=-2, dim2=-1).contiguous()
        

        mean_ret = self.ret_means
        if mean_ret.shape[1] < target_dims:
            mean_pad_dim = torch.zeros(mean_ret.shape[0], target_dims - mean_ret.shape[1],
                                       device=mean_ret.device, dtype=mean_ret.dtype)
            mean_ret = torch.cat([mean_ret, mean_pad_dim], dim=1) 
        if mean_ret.shape[0] < target_clusters:
            mean_pad_cluster = torch.zeros(target_clusters - mean_ret.shape[0], target_dims,
                                           device=mean_ret.device, dtype=mean_ret.dtype)
            mean_ret = torch.cat([mean_ret, mean_pad_cluster], dim=0)   
        self.ret_means_5x100 = mean_ret

        ret = self.ret_variances
        if ret.shape[1] < target_dims:
            pad_dim = torch.zeros(ret.shape[0], target_dims - ret.shape[1], device=ret.device, dtype=ret.dtype)
            ret = torch.cat([ret, pad_dim], dim=1) 
        if ret.shape[0] < target_clusters:
            pad_cluster = torch.zeros(target_clusters - ret.shape[0], target_dims, device=ret.device, dtype=ret.dtype)
            ret = torch.cat([ret, pad_cluster], dim=0)  
        self.ret_variances_5x100 = ret
    
        self.inf_ret_means = self.means.contiguous()
        self.inf_ret_variances = torch.diagonal(self.inflated_covariances, dim1=-2, dim2=-1).contiguous()
        
        inf_mean_ret = self.inf_ret_means
        if inf_mean_ret.shape[1] < target_dims:
            inf_mean_pad_dim = torch.zeros(inf_mean_ret.shape[0], target_dims - inf_mean_ret.shape[1],
                                       device=inf_mean_ret.device, dtype=inf_mean_ret.dtype)
            inf_mean_ret = torch.cat([inf_mean_ret, inf_mean_pad_dim], dim=1)
        if inf_mean_ret.shape[0] < target_clusters:
            inf_mean_pad_cluster = torch.zeros(target_clusters - inf_mean_ret.shape[0], target_dims,
                                           device=inf_mean_ret.device, dtype=inf_mean_ret.dtype)
            inf_mean_ret = torch.cat([inf_mean_ret, inf_mean_pad_cluster], dim=0) 
        self.inf_ret_means_5x100 = inf_mean_ret

        inf_ret = self.inf_ret_variances
        if inf_ret.shape[1] < target_dims:
            pad_dim = torch.zeros(inf_ret.shape[0], target_dims - inf_ret.shape[1], device=inf_ret.device, dtype=inf_ret.dtype)
            inf_ret = torch.cat([inf_ret, pad_dim], dim=1)
        if inf_ret.shape[0] < target_clusters:
            inf_pad_cluster = torch.zeros(target_clusters - inf_ret.shape[0], target_dims, device=inf_ret.device, dtype=inf_ret.dtype)
            inf_ret = torch.cat([inf_ret, inf_pad_cluster], dim=0)  
        self.inf_ret_variances_5x100 = inf_ret
        
        


    def draw_samples(self, num_samples, return_params=True):
        """
        Draws samples from the d-dimensional Gaussian Mixture Model.

        Parameters:
        num_samples (int): Number of samples to draw.
        return_params (bool): If True, also return per-sample means/covariances
                              and component ids.

        Returns:
        torch.Tensor: samples drawn from the GMM.
        If return_params is True, returns:
            samples (num_samples, d),
            sample_means (num_samples, d),
            sample_covariances (num_samples, d),  # diagonal variances only
            component_choices (num_samples,)
        """
        samples = torch.zeros(num_samples, self.d, device=self.device)
        sample_means = torch.zeros(num_samples, self.d, device=self.device) if return_params else None
        sample_covariances = torch.zeros(num_samples, self.d, device=self.device) if return_params else None
        component_choices = torch.multinomial(self.weights, num_samples, replacement=True)
        for cluster_id in range(self.num_cluster):
            mask = (component_choices == cluster_id)
            num_cluster_samples = mask.sum().item()
            if num_cluster_samples > 0:
                sample = self.GMM4sample[cluster_id].sample((num_cluster_samples,))
                samples[mask] = sample
                if return_params:
                    sample_means[mask] = self.means[cluster_id]
                    sample_covariances[mask] = torch.diagonal(self.covariances[cluster_id], dim1=-2, dim2=-1)
        if return_params:
            return samples, sample_means, sample_covariances, component_choices
        return samples


    def draw_inflated_samples(self, num_samples, return_params=True):
        """
        Draws samples from the d-dimensional inflated Gaussian Mixture Model.

        Parameters:
        num_samples (int): Number of samples to draw.

        Returns:
        torch.Tensor: samples drawn from the inflated GMM.
        """
        samples = torch.zeros(num_samples, self.d, device=self.device)
        component_choices = torch.multinomial(self.weights, num_samples, replacement=True)
        sample_means = torch.zeros(num_samples, self.d, device=self.device) if return_params else None
        sample_covariances = torch.zeros(num_samples, self.d, device=self.device) if return_params else None
        for cluster_id in range(self.num_cluster):
            mask = (component_choices == cluster_id)
            num_cluster_samples = mask.sum().item()
            if num_cluster_samples > 0:
                sample = self.GMM4inf[cluster_id].sample((num_cluster_samples,))  # .type_as(self.weights)
                samples[mask] = sample
                if return_params:
                    sample_means[mask] = self.means[cluster_id]
                    sample_covariances[mask] = torch.diagonal(self.covariances[cluster_id], dim1=-2, dim2=-1)
        if return_params:
            return samples, sample_means, sample_covariances, component_choices
        return samples


    def mahalanobis_distance(self, sample, mean, inv_covariance):
        """
        Computes the Mahalanobis distance of a sample from a given mean and inverse covariance matrix.

        Parameters:
        sample (torch.Tensor): Sample point. (d, )
        mean (torch.Tensor): Mean vector.  (d, )
        inv_covariance (torch.Tensor): Inverse covariance matrix.  (sub-dims, )

        Returns:
        float: Mahalanobis distance of the sample from the mean.
        """
        delta = sample[self.sub_dims] - mean[self.sub_dims]
        return torch.sqrt((delta @ inv_covariance @ delta).sum())

    def batched_squared_mahalanobis_distance(self, X, mean, inv_cov):
        delta = X[:, self.sub_dims] - mean[self.sub_dims]
        return torch.diag(delta @ inv_cov @ delta.T)


    def draw_inliers(self, num_samples, return_params=True):
        batch_size = max(num_samples * 2, 1000)
        samples = []
        sample_means = [] if return_params else None
        sample_covariances = [] if return_params else None
        total_samples_needed = num_samples
        while total_samples_needed > 0:
            raw_samples, raw_means, raw_covariances, _ = self.draw_samples(batch_size, return_params=True)
            batch_distances = self.get_squared_batched_dist(raw_samples)
            min_squared_distances = torch.min(batch_distances, dim=1).values
            inliner_mask = min_squared_distances < self.threshold_minus_delta
            selected_samples = raw_samples[inliner_mask]
            num_selected = selected_samples.shape[0]
            if num_selected > 0:
                if num_selected >= total_samples_needed:
                    samples.append(selected_samples[:total_samples_needed])
                    if return_params:
                        sample_means.append(raw_means[inliner_mask][:total_samples_needed])
                        sample_covariances.append(raw_covariances[inliner_mask][:total_samples_needed])
                    total_samples_needed = 0
                else:
                    samples.append(selected_samples)
                    if return_params:
                        sample_means.append(raw_means[inliner_mask])
                        sample_covariances.append(raw_covariances[inliner_mask])
                    total_samples_needed -= num_selected
        samples = torch.cat(samples)
        if return_params:
            sample_means = torch.cat(sample_means)
            sample_covariances = torch.cat(sample_covariances)
            return samples, sample_means, sample_covariances
        return samples

    def draw_local_anomalies(self, num_samples, return_params=True):
        batch_size = max(num_samples * 2, 1000)
        samples = []
        sample_means = [] if return_params else None
        sample_covariances = [] if return_params else None
        total_samples_needed = num_samples
        while total_samples_needed > 0:
            raw_samples, raw_means, raw_covariances, _ = self.draw_inflated_samples(batch_size, return_params=True)
            batch_distances = self.get_squared_batched_dist(raw_samples)
            min_squared_distances = torch.min(batch_distances, dim=1).values
            anomaly_mask = min_squared_distances > self.threshold_plus_delta
            selected_samples = raw_samples[anomaly_mask]
            num_selected = selected_samples.shape[0]
            if num_selected > 0:
                if num_selected >= total_samples_needed:
                    samples.append(selected_samples[:total_samples_needed])
                    if return_params:
                        sample_means.append(raw_means[anomaly_mask][:total_samples_needed])
                        sample_covariances.append(raw_covariances[anomaly_mask][:total_samples_needed])
                    total_samples_needed = 0
                else:
                    samples.append(selected_samples)
                    if return_params:
                        sample_means.append(raw_means[anomaly_mask])
                        sample_covariances.append(raw_covariances[anomaly_mask])
                    total_samples_needed -= num_selected
        samples = torch.cat(samples)
        if return_params:
            sample_means = torch.cat(sample_means)
            sample_covariances = torch.cat(sample_covariances)
            return samples, sample_means, sample_covariances
        return samples

    def assert_inliers(self, samples):
        for sample in samples:
            distances = [self.mahalanobis_distance(sample, mean, inv_cov) for mean, inv_cov in
                         zip(self.means, self.inv_sub_covariances)]
            assert min(distances) ** 2 < self.threshold_minus_delta

    def assert_local_anomalies(self, samples):
        for sample in samples:
            distances = [self.mahalanobis_distance(sample, mean, inv_cov) for mean, inv_cov in
                         zip(self.means, self.inv_sub_covariances)]
            assert min(distances) ** 2 > self.threshold_plus_delta

    def get_squared_batched_dist(self, raw_samples):
        batch_dist = []
        for mean, inv_cov in zip(self.means, self.inv_sub_covariances):
            distances = self.batched_squared_mahalanobis_distance(X=raw_samples, mean=mean, inv_cov=inv_cov)
            batch_dist.append(distances)
        return torch.stack(batch_dist, dim=1)  # (#samples, num_cluster)


    def draw_batched_data(self, num_inliers, num_local_anomalies, return_params=True):
        raw_inliers, ret_means, ret_covariances,_ = self.draw_samples(num_samples=int(num_inliers * 2), return_params=return_params)
        raw_local_anomalies,ret_anomaly_means, ret_anomaly_covariances,_ = self.draw_inflated_samples(num_samples=int(num_local_anomalies * 2),return_params=True)

        inliers_squared_dist = self.get_squared_batched_dist(raw_samples=raw_inliers)
        local_anomalies_squared_dist = self.get_squared_batched_dist(raw_samples=raw_local_anomalies)

        min_inliers_squared_dist = torch.min(inliers_squared_dist, dim=1).values
        min_local_anomalies_squared_dist = torch.min(local_anomalies_squared_dist, dim=1).values

        inliers_mask = min_inliers_squared_dist < self.threshold_minus_delta  # (#raw-inliers, )
        local_anomalies_mask = min_local_anomalies_squared_dist > self.threshold_plus_delta  # (#raw-la, )

        inliers = raw_inliers[inliers_mask][:num_inliers]
        local_anomalies = raw_local_anomalies[local_anomalies_mask][:num_local_anomalies]
        
        ret_means = ret_means[inliers_mask][:num_inliers]
        ret_covariances= ret_covariances[inliers_mask][:num_inliers]
        
        ret_anomaly_means = ret_anomaly_means[local_anomalies_mask][:num_local_anomalies]
        ret_anomaly_covariances = ret_anomaly_covariances[local_anomalies_mask][:num_local_anomalies]

        def add_extra(existing_samples, existing_means, existing_covariances, target_num_samples, draw_func):
            if existing_samples.shape[0] < target_num_samples:
                extra_samples, extra_means, extra_covariances = draw_func(num_samples=target_num_samples - existing_samples.shape[0], return_params=True)
                existing_samples = torch.concat([existing_samples, extra_samples], dim=0)
                existing_means = torch.concat([existing_means, extra_means], dim=0)
                existing_covariances = torch.concat([existing_covariances, extra_covariances], dim=0)
            return existing_samples, existing_means, existing_covariances

        inliers, ret_means, ret_covariances = add_extra(existing_samples=inliers, existing_means=ret_means, existing_covariances=ret_covariances, target_num_samples=num_inliers, draw_func=self.draw_inliers)
        local_anomalies, ret_anomaly_means, ret_anomaly_covariances = add_extra(existing_samples=local_anomalies, existing_means=ret_anomaly_means, existing_covariances=ret_anomaly_covariances, target_num_samples=num_local_anomalies,
                                    draw_func=self.draw_local_anomalies)
        if return_params:
            return inliers, local_anomalies, ret_means, ret_covariances, ret_anomaly_means, ret_anomaly_covariances
        return inliers, local_anomalies



def make_NdMclusterGMM(dim: int, num_cluster: int, weights: torch.Tensor, max_mean: int, max_var: int,
                       inflate_full: bool, device, sub_dims=None, percentile=0.80, delta=0.05):
    # Generate means between -max_mean and max_mean
    means = torch.rand(num_cluster, dim, device=device) * \
            torch.randint(low=-max_mean, high=max_mean+1, size=(num_cluster, dim, ), device=device)

    # Generate diagonal covariance matrices with positive entries between 1 and max_var
    diag_values = torch.rand(num_cluster, dim, device=device) * \
                  torch.randint(low=1, high=max_var+1, size=(num_cluster, dim, ), device=device)
    diag_values[diag_values == 0] = max_var / 2

    # Create batch of diagonal covariance matrices
    covariances = torch.diag_embed(diag_values)  # Shape: (num_cluster, dim, dim)

    N_d_M_cluster_gaussian = GaussianMixtureModel(
        means=means,
        covariances=covariances,
        weights=weights,
        inflate_full=inflate_full,
        sub_dims=sub_dims,
        percentile=percentile,
        delta=delta,
        device=device
    )
    return N_d_M_cluster_gaussian


def generate_constrained_eigenvals(d):
    # Generate uniformly distributed values in the range (-0.8, -0.2)
    low_range = np.random.uniform(-1.0, -0.1, size=d)

    # Generate uniformly distributed values in the range (0.2, 0.8)
    high_range = np.random.uniform(0.1, 1.0, size=d)

    # Randomly choose between the two ranges for each element
    choice = np.random.choice([0, 1], size=d)
    vector = np.where(choice == 0, low_range, high_range)

    return vector


def generate_full_rank_matrix(dim, device, scale=1):
    # Generate a random orthogonal matrix using QR decomposition
    A = np.random.rand(dim, dim)
    Q, _ = np.linalg.qr(A)

    eigenvals = generate_constrained_eigenvals(d=dim)
    eigenvals = np.diag(eigenvals)

    full_rank_matrix = Q @ eigenvals @ Q.T
    assert np.linalg.matrix_rank(full_rank_matrix) == dim
    if device is None:  # source is numpy
        return full_rank_matrix
    else:
        return torch.from_numpy(full_rank_matrix).to(dtype=torch.float, device=device)


def generate_linear_transform(dim, device, A_scale=1, b_scale=1):
    A = generate_full_rank_matrix(dim=dim, device=device, scale=A_scale)
    b = np.random.rand(dim) * np.random.randint(low=-b_scale, high=b_scale + 1, size=dim)  # [low, high)

    if device is not None:  # source is torch, transfer from numpy to torch
        b = torch.from_numpy(b).to(dtype=torch.float, device=device)
    return A, b


def transform_means(means, sub_dims, A, b):
    trans = []
    for mean in means:
        new_mean = mean.clone()
        new_mean[sub_dims] = A @ new_mean[sub_dims] + b
        trans.append(new_mean)
    return torch.stack(trans)


def transform_covs(covs, sub_dims, A):
    trans = []
    for cov in covs:
        new_cov = cov.clone()
        new_cov[sub_dims[:, None], sub_dims] = A @ new_cov[sub_dims[:, None], sub_dims] @ A.T
        trans.append(new_cov)
    return torch.stack(trans)


def transform_samples(samples, sub_dims, A, b, is_source_numpy=False):
    if is_source_numpy:
        new_samples = samples.copy()
    else:
        new_samples = samples.clone()

    if sub_dims is None:
        new_samples = new_samples @ A.T + b
    else:
        new_samples[:, sub_dims] = new_samples[:, sub_dims] @ A.T + b

    return new_samples

In [2]:

import numpy as np
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics  import roc_auc_score
from contextlib import contextmanager
import time, torch, random
import pandas as pd

def set_seed(seed: int = 0):
        # Python built-in RNG
    random.seed(seed)
    # NumPy RNG
    np.random.seed(seed)
    # PyTorch CPU RNG
    torch.manual_seed(seed)
    # PyTorch CUDA RNG (all GPUs)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Optional: make some operations deterministic (can slow things down)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
@contextmanager
def timer(name):
    t0 = time.perf_counter()
    yield
    print(f"{name}: {time.perf_counter()-t0:6.3f} s")


In [6]:
dimension_range = [(2,21),(21,41),(41,61),(61,81),(81,101)]
base_seed= 52324

for j in tqdm(range(len(dimension_range))):
    dimension = dimension_range[j]
    count = 0
    i = 0
    begin_idx = j * 32
    while count < 32:
        set_seed(base_seed + i)
        s = time.time()
        device = 'cuda:0'
        dim = np.random.randint(low=dimension[0], high=dimension[1])  # draw from [2, 20]
        num_cluster = np.random.randint(low=2, high=6)  # draw from [2, 5]
        max_mean = np.random.randint(low=2, high=6)  # draw from [2, 5]
        max_var = np.random.randint(low=2, high=6)  # draw from [2, 5]
        #print('num cluster', num_cluster)
        #print('dim', dim)
        model = make_NdMclusterGMM(dim=dim, num_cluster=num_cluster, weights=torch.tensor([1 / num_cluster] * num_cluster, device=device),
                                    max_mean=max_mean, max_var=max_var, inflate_full=False, sub_dims=None,
                                    percentile=0.9, delta=0.05, device=device)
        global_mean = model.ret_means_5x100
        global_variance = model.ret_variances_5x100
        global_anomaly_mean = model.inf_ret_means_5x100
        global_anomaly_variance = model.inf_ret_variances_5x100
        
        num_inliers = 5000
        num_outliers = 5000

        # test the batch data follows the inliner/local_anomaly criterion
        X_in, X_out, individual_means, individual_variances, anomaly_means, anomaly_variances\
                                      = model.draw_batched_data(num_inliers, num_outliers,return_params=True)            
                                      
        num_train = np.random.randint(500,4_000)
        #50%, 50% split among the rest
        num_test = 5000 - num_train
        num_inlier_test = int(0.5 * num_test)
        num_outlier_test = num_test - num_inlier_test
        
        perm = torch.randperm(X_in.size(0), device=X_in.device)
        train_idx = perm[:num_train]
        inlier_test_idx = perm[num_train:num_train + num_inlier_test]
        X_train = X_in[train_idx]
        X_inlier_test = X_in[inlier_test_idx]
        mean_train = individual_means[train_idx]
        variance_train = individual_variances[train_idx]
        mean_test_inlier = individual_means[inlier_test_idx]
        variance_test_inlier = individual_variances[inlier_test_idx]
        
        perm = torch.randperm(X_out.size(0), device=X_out.device)
        test_idx = perm[:num_outlier_test]
        X_outlier_test = X_out[test_idx] 
        mean_test_outlier = anomaly_means[test_idx]
        variance_test_outlier = anomaly_variances[test_idx]
        
        X_test = torch.cat([X_inlier_test, X_outlier_test], dim= 0)
        y_test = torch.cat([
            torch.zeros(X_inlier_test.size(0), device=X_inlier_test.device),
            torch.ones(X_outlier_test.size(0), device=X_outlier_test.device)
        ], dim=0)
        mean_test = torch.cat([mean_test_inlier, mean_test_outlier], dim = 0)
        variance_test = torch.cat([variance_test_inlier, variance_test_outlier],dim=0)
        
        saved = begin_idx + count
        np.savez(
                f"/home/xding2/FoMo-Meta/synthetic_data/gaussian_inliers/gaussian_{saved}.npz", 
                X_train = X_train.detach().cpu().numpy(),
                y_train = torch.zeros(X_train.shape[0]).detach().cpu().numpy(),
                X_test = X_test.detach().cpu().numpy(),
                y_test = y_test.detach().cpu().numpy(),
                mean_train = mean_train.detach().cpu().numpy(),
                variance_train = variance_train.detach().cpu().numpy(),
                mean_test = mean_test.detach().cpu().numpy(),
                variance_test = variance_test.detach().cpu().numpy(),
                global_mean = global_mean.detach().cpu().numpy(),
                global_variance = global_variance.detach().cpu().numpy(),
                global_anomaly_mean = global_anomaly_mean.detach().cpu().numpy(),
                global_anomaly_variance = global_anomaly_variance.detach().cpu().numpy(),
                )
        count += 1
        i += 1

    print(f"For dimension {dimension[0]}, {dimension[1]}. Total accepted ratio: ", (count / i ))

 20%|██        | 1/5 [00:00<00:02,  1.90it/s]

For dimension 2, 21. Total accepted ratio:  1.0


 40%|████      | 2/5 [00:01<00:01,  1.90it/s]

For dimension 21, 41. Total accepted ratio:  1.0


 60%|██████    | 3/5 [00:01<00:01,  1.78it/s]

For dimension 41, 61. Total accepted ratio:  1.0


 80%|████████  | 4/5 [00:02<00:00,  1.64it/s]

For dimension 61, 81. Total accepted ratio:  1.0


100%|██████████| 5/5 [00:03<00:00,  1.62it/s]

For dimension 81, 101. Total accepted ratio:  1.0


In [6]:
# import numpy as np
# import torch
# import matplotlib.pyplot as plt

# def to_numpy(X):
#     if isinstance(X, torch.Tensor):
#         return X.detach().cpu().numpy()
#     return X

# def visualize_in_out_2d(X_in, X_out, max_points=5000):
#     X_in = to_numpy(X_in)
#     X_out = to_numpy(X_out)
#     assert X_in.shape[1] == 2 and X_out.shape[1] == 2, "X must be 2D"

#     n_in, n_out = len(X_in), len(X_out)
#     rng = np.random.default_rng(42)
#     idx_in = rng.choice(n_in, size=min(n_in, max_points), replace=False)
#     idx_out = rng.choice(n_out, size=min(n_out, max_points), replace=False)
#     X_in_s = X_in[idx_in]
#     X_out_s = X_out[idx_out]

#     plt.figure(figsize=(7,6))
#     plt.scatter(X_in_s[:,0], X_in_s[:,1], s=6, alpha=0.6, label='inliers', c='#1f77b4')
#     plt.scatter(X_out_s[:,0], X_out_s[:,1], s=6, alpha=0.6, label='outliers', c='#d62728')
#     plt.title('Raw 2D scatter: X_in vs X_out')
#     plt.xlabel('X[:,0]'); plt.ylabel('X[:,1]'); plt.legend(); plt.tight_layout()
#     plt.show()

# # Call with your tensors
# for i in range(5):
#     device = 'cuda:0'
#     dim = np.random.randint(low=2, high=3)  # draw from [2, 20]
#     num_cluster = np.random.randint(low=2, high=6)  # draw from [2, 5]
#     max_mean = np.random.randint(low=2, high=6)  # draw from [2, 5]
#     max_var = np.random.randint(low=2, high=6)  # draw from [2, 5]
#     #print('num cluster', num_cluster)
#     #print('dim', dim)
#     model = make_NdMclusterGMM(dim=dim, num_cluster=num_cluster, weights=torch.tensor([1 / num_cluster] * num_cluster, device=device),
#                                 max_mean=max_mean, max_var=max_var, inflate_full=False, sub_dims=None,
#                                 percentile=0.9, delta=0.05, device=device)

#     num_inliers =5000
#     num_outliers =5000
#     X_in, X_out = model.draw_batched_data(num_inliers, num_outliers)
#     visualize_in_out_2d(X_in, X_out)

In [ ]:
# dimension_range = [(2,21),(21,41),(41,61),(61,81),(81,101),(101,151),(151,251)]
# base_seed= 1024 #52324

# for j in tqdm(range(len(dimension_range))):
#     dimension = dimension_range[j]
#     count = 0
#     i = 0
#     begin_idx = j * 32
#     while count < 32:
#         set_seed(base_seed + i)
#         s = time.time()
#         device = 'cuda:0'
#         dim = np.random.randint(low=dimension[0], high=dimension[1])  # draw from [2, 20]
#         num_cluster = np.random.randint(low=2, high=6)  # draw from [2, 5]
#         max_mean = np.random.randint(low=2, high=6)  # draw from [2, 5]
#         max_var = np.random.randint(low=2, high=6)  # draw from [2, 5]
#         #print('num cluster', num_cluster)
#         #print('dim', dim)
#         model = make_NdMclusterGMM(dim=dim, num_cluster=num_cluster, weights=torch.tensor([1 / num_cluster] * num_cluster, device=device),
#                                     max_mean=max_mean, max_var=max_var, inflate_full=False, sub_dims=None,
#                                     percentile=0.9, delta=0.05, device=device)

#         num_inliers = np.random.randint(1_000, 5_000)
#         outliers_ratio = np.random.uniform(0.05, 0.15)
#         num_outliers = int(outliers_ratio * num_inliers)


#         #print(f'drawing {num_inliers} inliers and {num_outliers} outliers')

#         # test the batch data follows the inliner/local_anomaly criterion
#         X_in, X_out = model.draw_batched_data(num_inliers, num_outliers)

#         X   = torch.cat([X_in, X_out]).cpu().numpy()
#         y   = np.concatenate([np.zeros(num_inliers), np.ones(num_outliers)])  # 1 ⇒ outlier

#         contamination = num_outliers / (num_inliers + num_outliers)
#        # with timer("LOF fit+score         "):
#         lof = LocalOutlierFactor(n_neighbors=20,
#                                         contamination=contamination)
#         _   = lof.fit_predict(X)                      # triggers fit
#         scores = -lof.negative_outlier_factor_       # higher ⇒ more anomalous
#         auc = roc_auc_score(y, scores)
#         #print(f"ROC-AUC (LOF vs truth): {auc:.4f}")

#         if auc < 0.95:
#             saved = begin_idx + count
#             print('num_cluster', num_cluster, 'dim', dim, 'saved idx', saved, 'auc', auc)
#             np.savez(f"synthetic_prior/val_data/gaussian/gaussian_{saved}.npz", X=X, y=y)
#             count += 1
#         i += 1

#     print(f"For dimension {dimension[0]}, {dimension[1]}. Total accepted ratio: ", (count / i ))

  0%|          | 0/7 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [16]:
import random, time, math, numpy as np
from contextlib import contextmanager

import torch
from torch.distributions import Normal, Beta, Exponential, StudentT
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics  import roc_auc_score
import scipy
from scipy.stats import beta as scipy_beta
import torch, math, multiprocessing as mp, numpy as np
import scipy.stats as sps
from torch.distributions import Normal
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.special import betainc, betaincinv


import time
import torch

# Make sure to sync for accurate GPU timings
def tic():
    torch.cuda.synchronize()
    return time.time()

def toc(start, label=""):
    torch.cuda.synchronize()
    end = time.time()
    print(f"{label}: {end - start:.6f} seconds")

# ───────────────────────── helper: icdf() calculation on torch ──────────────────────────
def _neg_log1p(u):
    # stable for u≈0
    return (-torch.log1p(-u)).clamp_min(0.0)


def exp_icdf(u, rate):
    """
    u     : (...,)  in (0,1)
    rate  : broadcastable with u   (λ > 0)
    """
    return _neg_log1p(u) / rate


def beta_icdf(p, a, b, n_grid: int = 1000):
    p = torch.as_tensor(p, dtype=torch.float32)
    a = torch.as_tensor(a, dtype=torch.float32)
    b = torch.as_tensor(b, dtype=torch.float32)
    if a.size() == torch.Size([]):
      a = a.expand(1)
    if b.size() == torch.Size([]):
      b = b.expand(1)
    x = torch.linspace(0.0, 1.0, n_grid + 1, dtype=torch.float32, device=p.device)
    mid = (x[:-1] + x[1:]) / 2.0
    mid = mid.view(-1, 1)  # shape [n_grid, 1]
    dx = 1.0 / n_grid

    log_norm = torch.lgamma(a) + torch.lgamma(b) - torch.lgamma(a + b)
    pdf_mid = torch.exp((a - 1) * torch.log(mid) +
                        (b - 1) * torch.log1p(-mid) -
                        log_norm)  # [n_grid, D]
    cdf = torch.cumsum(pdf_mid, dim=0) * dx  # [n_grid, D]
    cdf = torch.cat([torch.zeros(1, cdf.shape[1], device=p.device), cdf], dim=0)  # [n_grid+1, D]
    # Expand p to match shape [N, D]
    if p.ndim == 1:
        p = p.unsqueeze(1)
    if a.ndim == 1:
        a = a.unsqueeze(0)
        b = b.unsqueeze(0)
    if p.shape[1] != a.shape[1]:
        p = p.expand(-1, a.shape[1])
    # Searchsorted per-dimension
    N, D = p.shape
    idx = torch.empty((N, D), dtype=torch.long, device=p.device)
    for d in range(D):
        idx[:, d] = torch.searchsorted(cdf[:, d], p[:, d], right=False).clamp(1, n_grid)
    # Gather x and y for interpolation
    x0 = x[idx - 1]
    x1 = x[idx]
    y0 = torch.empty_like(p)
    y1 = torch.empty_like(p)
    for d in range(D):
        y0[:, d] = cdf[idx[:, d] - 1, d]
        y1[:, d] = cdf[idx[:, d], d]

    t = (p - y0) / (y1 - y0 + 1e-12)
    return torch.clamp(x0 + t * (x1 - x0), 0., 1.)


def student_t_icdf(u, df):
    u = torch.as_tensor(u, dtype=torch.float32)
    df = torch.as_tensor(df, dtype=torch.float32)

    p = 2.0 * torch.minimum(u, 1 - u)
    p = p.to(df.device)
    x = beta_icdf(p, 0.5 * df, torch.tensor(0.5,device = p.device))
    t = torch.sqrt(df * (1.0 / x - 1.0))
    return torch.where(u > 0.5, t, -t)


def normal_cdf(x):  
    return 0.5*(1+torch.special.erf(x/math.sqrt(2)))


def normal_ppf(u):  
    return Normal(0,1).icdf(u)


# ─────────────── low-level helpers ───────────────
def rand_corr_batch(batch, d, identity=False,device='cuda'):
    """(B,d,d) stack of correlation matrices on device/torch.float32"""
    if identity:
        eye = torch.eye(d, device=device, dtype=torch.float32)
        return eye.expand(batch, -1, -1).clone()

    A   = torch.rand(batch, d, d, device=device, dtype=torch.float32)
    psd = A @ A.transpose(-1, -2)
    diag= torch.diagonal(psd, dim1=-2, dim2=-1)
    norm= torch.sqrt(torch.clamp(diag, 1e-12))
    C   = psd / (norm.unsqueeze(-1)*norm.unsqueeze(-2))
    C.diagonal(dim1=-2, dim2=-1).fill_(1.)
    C += torch.eye(d, device=device, dtype=torch.float32).unsqueeze(0)*1e-6
    return C


def mvn_sample(chol,device):       # chol: (B,d,d) or (d,d)
    if chol.dim() == 2: chol = chol.unsqueeze(0)
    B,d = chol.shape[:2]
    z   = torch.randn(B, d, 1, device=device, dtype=torch.float32)
    return (chol @ z).squeeze(-1)           # (B,d)


# ─────────────── mixture helpers ───────────────
class GaussianMix:
    def __init__(self, comps, device='cpu'):
        w, mu, sigma = zip(*comps)
        self.device = device
        self.w = torch.tensor(w, device=device, dtype=torch.float32)
        self.mu = torch.tensor(mu, device=device, dtype=torch.float32)
        self.sigma = torch.tensor(sigma, device=device, dtype=torch.float32)
        self.w /= self.w.sum()

    def cdf(self, x):
        x = torch.as_tensor(x, dtype=torch.float32, device=self.device)
        z = (x.unsqueeze(-1) - self.mu) / (self.sigma * math.sqrt(2))
        return (self.w * (0.5 * (1 + torch.erf(z)))).sum(-1)

    def ppf_bounds(self):
        lo = (self.mu - 5 * self.sigma).min().item()
        hi = (self.mu + 5 * self.sigma).max().item()
        return lo, hi

    def ppf(self, u, tol=1e-5, max_iter=100):
        """
        Numerical inverse CDF using bisection.
        u: tensor of shape (...)
        Returns: tensor of same shape
        """
        u = torch.as_tensor(u, device=self.device, dtype=torch.float32)
        low, high = self.ppf_bounds()
        low = torch.full_like(u, low)
        high = torch.full_like(u, high)

        for _ in range(max_iter):
            mid = 0.5 * (low + high)
            cdf_mid = self.cdf(mid)
            low = torch.where(cdf_mid < u, mid, low)
            high = torch.where(cdf_mid >= u, mid, high)
            if torch.max(high - low) < tol:
                break
        return 0.5 * (low + high)


class BetaMix:
    def __init__(self, comps,device='cuda'):
        self.comps = comps
        self.w = torch.tensor([c[0] for c in comps], device=device, dtype=torch.float32)
        self.a = torch.tensor([c[1] for c in comps], device=device, dtype=torch.float32)
        self.b = torch.tensor([c[2] for c in comps], device=device, dtype=torch.float32)
        self.loc = torch.tensor([c[3] for c in comps], device=device, dtype=torch.float32)
        self.sc = torch.tensor([c[4] for c in comps], device=device, dtype=torch.float32)
        self.w /= self.w.sum()
        self.device = device

    def cdf(self, x: torch.Tensor):
        # Fallback to CPU-based computation
        x_cpu = x.detach().cpu().numpy()
        cdf_vals = np.zeros_like(x_cpu)
        for w, a, b, loc, scale in zip(self.w, self.a, self.b, self.loc, self.sc):
            dist = scipy_beta(a=a.item(), b=b.item(), loc=loc.item(), scale=scale.item())
            cdf_vals += w.item() * dist.cdf(x_cpu)
        return torch.tensor(cdf_vals, device=self.device, dtype=torch.float32)

    def ppf_bounds(self):
        lo = (self.loc).min().item()
        hi = (self.loc + self.sc).max().item()
        return lo, hi

# ─────────────── marginal catalogue ───────────────
def rand_def(device='cuda',
             PPF_GRID = 1_000):
    cat = random.choice(["normal","beta","beta_mix","expo","gauss_mix","beta_mix","student"])
    if cat=="normal":
        return dict(kind="torch", dist=Normal(torch.tensor(random.uniform(-1,1)),
                                              torch.tensor(random.uniform(1.5,2))))
    if cat=="beta":
        return dict(kind="torch", dist=Beta(torch.tensor(random.uniform(1,5)),
                                            torch.tensor(random.uniform(1,5))))
    if cat=="expo":
        return dict(kind="torch", dist=Exponential(torch.tensor(random.uniform(1,2))))
    if cat=="student":
        return dict(kind="torch", dist=StudentT(torch.tensor(random.randint(3,10))))
    if cat=="gauss_mix":
        comps = [(random.uniform(0.3,0.7),
                  random.uniform(-3,3),
                  random.uniform(0.5,1.5)) for _ in range(random.randint(1,3))]
        mix   = GaussianMix(comps)
    else:
        comps = [(random.uniform(0.3,0.7),
                  random.uniform(1,5),
                  random.uniform(1,5),
                  -5.0, 10.0) for _ in range(random.randint(1,3))]
        mix   = BetaMix(comps,device=device)
    # build interpolation grids
    u_grid = torch.linspace(0.001,0.999,PPF_GRID, device=device, dtype=torch.float32)
    lo,hi  = mix.ppf_bounds()
    x_grid = torch.linspace(lo,hi,PPF_GRID, device=device, dtype=torch.float32)
    #cdf_vals = mix.cdf(x_grid)
    return dict(kind="interp", u=u_grid, x=x_grid, lo=lo, hi=hi)


class CorpulaGenerator:
    def __init__(self, num_dims, device="cuda", ppf_grid=2_000):
        self.num_dims = num_dims
        self.device = device
        self.dtype = torch.float32
        self.ppf_grid = ppf_grid
        self.chol_base = torch.linalg.cholesky(
            rand_corr_batch(1, num_dims, device=device)[0]
        )
        self.specs = [rand_def(device=device, PPF_GRID=ppf_grid) for _ in range(num_dims)]


    def sample_inliers(self, num_inliers):
        """Sample clean multivariate normal data & transform to marginals"""
        z = torch.randn(num_inliers, self.num_dims, device=self.device, dtype=self.dtype)
        samples = (z @ self.chol_base.T)
        return samples

    def sample_outliers(self,
                        num_outliers,
                        method="perturb_u_values",
                        strength=0.2):
        """Generate outliers using perturb-u or disturbed covariance"""
        if method == "perturb_u_values":
            # 1. Sample from base MVN
            z = torch.randn(num_outliers, self.num_dims, device=self.device, dtype=self.dtype)
            samples = (z @ self.chol_base.T)

            # 2. Transform to uniform
            U = normal_cdf(samples)
            min_k = max(1, math.ceil(0.02 * self.num_dims))
            max_k = max(min_k, math.floor(0.2 * self.num_dims))

            # Ensure min_k < max_k + 1 to avoid invalid range
            if min_k >= max_k + 1:
                max_k = min_k # fallback: both min_k and max_k equal, will result in k_row = min_k

            k_row = torch.randint(min_k, max_k + 1, (num_outliers,), device=self.device)
            #k_row = torch.randint(5, 6, (num_outliers,), device=self.device) 
            perm = torch.rand(num_outliers, self.num_dims, device=self.device).argsort(dim=1)
            sel_mask = torch.arange(self.num_dims, device=self.device).expand(num_outliers, -1) < k_row.unsqueeze(1)
            mask = torch.zeros_like(U, dtype=torch.bool).scatter(1, perm, sel_mask)

            push0 = torch.rand_like(U) < 0.5
            z_mask, o_mask = mask & push0, mask & ~push0
            noise = torch.empty_like(U)
            noise[z_mask] = strength * torch.rand(z_mask.sum(), device=self.device) * 0.5
            noise[o_mask] = 1.0 - strength * torch.rand(o_mask.sum(), device=self.device) * 0.5
            U[mask] = noise[mask]

            samples = Normal(0, 1).icdf(U)
            return samples

        elif method == "disturb_covariance":
            d, device = self.num_dims, self.device
            base_L    = self.chol_base                       
            # 1) Random block lengths k  and start indices i0  –– now 1 … d inclusive 
            lowerbound = int(1+ d//3)
            upperbound = min(int(1+ 2 * d//3),d+1)
            if lowerbound == upperbound:
              upperbound += 1
            k      = torch.randint(lowerbound, upperbound, (num_outliers,), device=device)     # (B,) #int(1+ d//2)
            k_max  = int(k.max())

            # Vectorised start positions: i0[b] ∈ {0 … d-k[b]}
            # torch.randint can’t take a per-element “high”, so we synthesise i0 using rand():
            max_start = d - k                          # (B,)
            i0 = (torch.rand(num_outliers, device=device) * (max_start + 1)
                ).floor().long()                      # (B,)

            i1 = i0 + k                                # (B,)  exclusive end index
            L_rand_full = torch.linalg.cholesky(
                rand_corr_batch(num_outliers, k_max, identity=True, device=device)          # (B,k_max,k_max)
            )
            rows  = torch.arange(k_max, device=device).view(1, k_max, 1)     # (1,k_max,1)
            cols  = torch.arange(k_max, device=device).view(1, 1, k_max)     # (1,1,k_max)
            keep  = (rows < k.view(-1, 1, 1)) & (cols < k.view(-1, 1, 1))    # (B,k_max,k_max)

            L_rand_pad = torch.zeros(num_outliers, d, d, device=device)      # (B,d,d)
            L_rand_pad[:, :k_max, :k_max][keep] = L_rand_full[keep]
            L_mix = base_L.expand(num_outliers, -1, -1).clone()

            row = torch.arange(d, device=device).view(1, d)        # (1,d)
            mask_rows = (row >= i0.view(-1, 1)) & (row < i1.view(-1, 1))  # (B,d)
            col = torch.arange(d, device=device).view(1, 1, d)     # (1,1,d)
            mask = mask_rows.unsqueeze(-1) & (col < row.unsqueeze(-1) + 1)

            L_mix = torch.where(mask,
                                (1- strength) * L_mix + strength *L_rand_pad,
                                L_mix)
            L_mix.diagonal(dim1=-2, dim2=-1).clamp_(min=1e-6)
            return mvn_sample(L_mix, device=device)  
        else:
            raise ValueError(f"Unsupported outlier injection method: {method}")

    def _transform(self, samples):
        """
        Fully GPU-based transform.
        • Normal, Beta, Exp, StudentT → vectorized GPU code
        • Mixture (interp) → fast interpolation
        """
        U = normal_cdf(samples)
        eps = 1e-6
        U = torch.clamp(U, eps, 1-eps) #make sure U does not approch infty
        N, D = U.shape
        X = torch.empty_like(U)
        # Group columns by distribution type
        interp_cols, normal_cols, beta_cols, exp_cols, student_cols = [], [], [], [], []
        for d, spec in enumerate(self.specs):
            if spec["kind"] == "interp":
                interp_cols.append(d)
            elif spec["kind"] == "torch":
                dist = spec["dist"]
                if isinstance(dist, Normal):
                    normal_cols.append(d)
                elif isinstance(dist, Beta):
                    beta_cols.append(d)
                elif isinstance(dist, Exponential):
                    exp_cols.append(d)
                elif isinstance(dist, StudentT):
                    student_cols.append(d)
        # ---------- Interp mixture sampling ----------
        for d in interp_cols:
            u = U[:, d]
            spec = self.specs[d]
            grid_min = spec["u"][0]
            grid_max = spec["u"][-1]
            n_bins = len(spec["u"]) - 1
            bin_width = (grid_max - grid_min) / n_bins
            # Clamp u to grid range
            u_clamped = torch.clamp(u, grid_min, grid_max - 1e-6)
            # Fast approximate bin index (assumes linear CDF grid)
            idx = ((u_clamped - grid_min) / bin_width).long().clamp(0, n_bins - 1)
            idx = torch.clamp(idx, 1, len(spec["u"]) - 1)
            u_lo, u_hi = spec["u"][idx - 1], spec["u"][idx]
            x_lo, x_hi = spec["x"][idx - 1], spec["x"][idx]
            X[:, d] = x_lo + (u - u_lo) * (x_hi - x_lo) / (u_hi - u_lo)
        # ---------- Normal ----------
        if normal_cols:
            loc = torch.tensor([self.specs[d]["dist"].loc for d in normal_cols],
                            device=self.device).view(1, -1)
            scale = torch.tensor([self.specs[d]["dist"].scale for d in normal_cols],
                                device=self.device).view(1, -1)
            dist = Normal(loc, scale)
            X[:, normal_cols] = dist.icdf(U[:, normal_cols])
        # ---------- Beta ----------
        if beta_cols:
            a = torch.tensor([self.specs[d]["dist"].concentration1 for d in beta_cols],
                            device=self.device).view(1, -1)
            b = torch.tensor([self.specs[d]["dist"].concentration0 for d in beta_cols],
                            device=self.device).view(1, -1)
            X[:, beta_cols] = beta_icdf(U[:, beta_cols], a, b)
        # ---------- Exponential ----------
        if exp_cols:
            rate = torch.tensor([self.specs[d]["dist"].rate for d in exp_cols],
                                device=self.device).view(1, -1)
            X[:, exp_cols] = exp_icdf(U[:, exp_cols], rate)
        # ---------- StudentT ----------
        if student_cols:
            df = torch.tensor([self.specs[d]["dist"].df for d in student_cols],
                            device=self.device).view(1, -1)
            X[:, student_cols] = student_t_icdf(U[:, student_cols], df)
        return X



    @torch.no_grad()
    def draw_batched_data(self,
                          num_inliers,
                          num_local_anomalies):
        METHOD = random.choice(['disturb_covariance']) #["disturb_covariance"])#,"perturb_u_values"])  # or add "perturb_u_values"
        STRENGTH = random.uniform(0.2,0.4) if METHOD == 'perturb_u_values' else random.uniform(0.97,0.99)
        inliers = self.sample_inliers(num_inliers)

        outliers =  self.sample_outliers(num_local_anomalies, method=METHOD, strength=STRENGTH)
        combined = torch.cat([inliers, outliers], dim=0)

        X_combined = self._transform(combined)
        X_inliers = X_combined[:num_inliers]
        X_outliers = X_combined[num_inliers:]

        return X_inliers, X_outliers


def make_corpula(device, 
                 max_feature_dim=100,
                 min_feature_dim = 2,
                 dim = None):
    if dim is not None:
        num_features = dim
    else:
        num_features = np.random.randint(min_feature_dim, max_feature_dim)
    return CorpulaGenerator(num_dims=num_features, device=device)


In [17]:
dimension_range = [(2,21),(21,41),(41,61),(61,81),(81,101),(101,151),(151,251)]
base_seed= 52324 #1024 #52324


for j in tqdm(range(len(dimension_range))):
    dimension = dimension_range[j]
    avg_auroc = 0.0
    count = 0
    i = 0
    begin_idx = j * 32
    while count < 32:
        set_seed(base_seed + i)
        s = time.time()
        device = 'cuda:0'
        dim = np.random.randint(low=dimension[0], high=dimension[1])  # draw from [2, 20]
        num_inliers = np.random.randint(1_000, 5_000)
        outliers_ratio = np.random.uniform(0.05, 0.15)
        num_outliers = int(outliers_ratio * num_inliers)
        gen = CorpulaGenerator(num_dims=dim, device=device)

        # test the batch data follows the inliner/local_anomaly criterion
        X_in, X_out = gen.draw_batched_data(num_inliers, num_outliers)

        X   = torch.cat([X_in, X_out]).cpu().numpy()
        y   = np.concatenate([np.zeros(num_inliers), np.ones(num_outliers)])  # 1 ⇒ outlier

        contamination = num_outliers / (num_inliers + num_outliers)
        lof = LocalOutlierFactor(n_neighbors=20,
                                        contamination=contamination)
        _   = lof.fit_predict(X)                  
        scores = -lof.negative_outlier_factor_   
        auc = roc_auc_score(y, scores)
        #print(auc)

        if auc > 0.5 and auc < 0.95:
            saved = begin_idx + count
            #print('num_cluster', num_cluster, 'dim', dim, 'saved idx', saved, 'auc', auc)
            #np.savez(f"synthetic_prior/test_data/copula_disturb/copuladisturb_{saved}.npz", X=X, y=y)
            count += 1
            avg_auroc += auc
        i += 1

    print(f"For dimension {dimension[0]}, {dimension[1]}. Total accepted ratio: ", (count / i ), " avg auc: ", (avg_auroc / count))

  0%|          | 0/7 [00:00<?, ?it/s]

 14%|█▍        | 1/7 [00:07<00:46,  7.70s/it]

For dimension 2, 21. Total accepted ratio:  0.9411764705882353  avg auc:  0.7826942541220995


 29%|██▊       | 2/7 [00:16<00:40,  8.10s/it]

For dimension 21, 41. Total accepted ratio:  1.0  avg auc:  0.8127244167077211


...

For dimension 81, 101. Total accepted ratio:  1.0  avg auc:  0.7894213136804981


 86%|████████▌ | 6/7 [01:42<00:23, 23.40s/it]

For dimension 101, 151. Total accepted ratio:  1.0  avg auc:  0.7856288374302677


100%|██████████| 7/7 [02:46<00:00, 23.78s/it]

For dimension 151, 251. Total accepted ratio:  1.0  avg auc:  0.7650654414147321


In [3]:
# UPDATED SCM prob and SCM contextual!!

import torch
import torch.nn as nn
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score
import time
import math
import random

def lognormal_discrete(mu,
                       sigma,
                       minval:int,
                       maxval:int):
    # sample from lognormal distribtuion, making it discrete
    # input: mu, sigma, minval (int), maxval(int)
    # return: a integer value
    val = int(np.round(np.random.lognormal(mu, sigma)))
    return int(np.clip(val, minval, maxval))


def sample_layers_and_nodes(min_num_layer =2,
                            max_num_layer =5,
                            min_hidden_size = 3,
                            max_hidden_size = 8):
    #return: a randomly sampled hidden layer and number of layers
    l = lognormal_discrete(mu=0.7, sigma=0.4, minval=min_num_layer, maxval=max_num_layer)  # num layers
    h = lognormal_discrete(mu=1.2, sigma=0.5, minval=min_hidden_size, maxval=max_hidden_size)  # hidden size
    return l, h

@torch.no_grad()
def sample_noise_distribution(device='cpu'):
    mu = (torch.rand(1, device=device) - 0.5).item()
    sigma = (torch.rand(1, device=device) * (0.5 - 0.05) + 0.05).item()
    def noise_func(n):
        return torch.exp(mu + sigma * torch.randn(n, device=device))
    noise_func.mu = mu
    noise_func.sigma = sigma
    return noise_func

@torch.no_grad()
def sample_activation(device='cpu'):
    # sample activation functions (PyTorch version)
    activations = [
        ("tanh", torch.tanh),
        ("leaky_relu", lambda x: torch.where(x > 0, x, 0.01 * x)),
        ("elu", lambda x: torch.where(x > 0, x, torch.exp(x) - 1)),
        ("identity", lambda x: x),
    ]
    idx = torch.randint(0, len(activations), (1,), device=device).item()
    return activations[idx]


@torch.no_grad()
def random_noise_scales_per_sample(num_samples, 
                                   layer_sizes, 
                                   chosen_nodes,
                                   high_noise=5.0, 
                                   high_noise_prob=0.2, 
                                   device='cpu'):
    """
    Generate random noise scales for each sample and node, for each layer.
    Returns: List of tensors, each shape (num_samples, layer_size)
    """
    perturb_prob_layer = high_noise_prob
    noise_scales = []
    # num_layers = len(layer_sizes)
    # min_prob = 0.0
    # max_prob = high_noise_prob
    # noise_scales = []
    for l, n in enumerate(layer_sizes):
    #     if num_layers > 1 and len(chosen_nodes) > 50:
    #         perturb_prob_layer = min_prob + (max_prob - min_prob) * (l / (num_layers - 1))
    #     else:
    #         perturb_prob_layer = max_prob  # single-layer edge case
        noise_scales.append(
        torch.where(
            torch.rand(num_samples, n, device=device) < perturb_prob_layer,
            torch.full((num_samples, n), high_noise, device=device),
            torch.ones(num_samples, n, device=device)
            )
        )
    return noise_scales


@torch.no_grad()
def create_weight_mask(
    num_samples, layers, chosen_nodes, perturb_prob=0.5, device='cpu'
):
    """
    Vectorized version: No per-sample loop.
    For each sample and layer, ensures that at least one parent of a chosen node is perturbed.
    """
    masks = []
    node_layer_size = layers[0].weight.shape[0]  # assumes all layers same size
    num_layers = len(layers)
    min_prob = 0.0
    max_prob = perturb_prob

    for l, layer in enumerate(layers):
        weight = layer.weight  # (out_features, in_features)
        perturbable = (torch.abs(weight) > 0.05)  # (out, in)
        mask = torch.ones(num_samples, *weight.shape, device=device)
        
        # Interpolate perturbation prob for this layer
        if num_layers > 1 and len(chosen_nodes) > 50:
            perturb_prob_layer = min_prob + (max_prob - min_prob) * (l / (num_layers - 1))
        else:
            perturb_prob_layer = max_prob  # single-layer edge case

        perturb_mask = (torch.rand(num_samples, *weight.shape, device=device) < perturb_prob_layer) & perturbable.unsqueeze(0)
        flip_mask = torch.rand(num_samples, *weight.shape, device=device) < 0.5

        mask[perturb_mask & flip_mask] = -1.0
        mask[perturb_mask & (~flip_mask)] = 0.0
        masks.append(mask)
        #check if all the masks in mask is 1
    return masks



class MaskedLinear(nn.Linear):
    def __init__(self, in_features, out_features, min_abs=0.35, device='cpu'):
        super().__init__(in_features, out_features, False)
        with torch.no_grad():
            w = torch.normal(mean=0., std=1., size=self.weight.shape, device=device)
            self.weight.copy_(w) #_clipped)
        self.mask = nn.Parameter(torch.ones_like(self.weight), requires_grad=False)


    def set_random_mask(self, keep_prob=0.7):
        with torch.no_grad():
            self.mask[:] = (torch.rand_like(self.mask) < keep_prob).float()

    def forward(self, input, weight_mask=None):
        masked_weight = self.weight * self.mask
        if weight_mask is None:
            return nn.functional.linear(input, masked_weight, None)
        batch = input.size(0)
        masked_weight = masked_weight.unsqueeze(0)  # (1, out_features, in_features)
        weight = masked_weight.expand(batch, -1, -1) * weight_mask  # (batch, out_features, in_features)
        out = torch.bmm(input.unsqueeze(1), weight.transpose(1,2)).squeeze(1)
        return out




class SCM_MLP(nn.Module):
    def __init__(self, hidden_dim, num_layers, activations,device='cuda'):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(MaskedLinear(hidden_dim, hidden_dim,device=device))
        assert len(activations) == len(self.layers)
        self.activations = activations
        
        # Per-node noise distributions for each layer and neuron
        self.noise_funcs = [
            [sample_noise_distribution(device) for _ in range(hidden_dim)]  # per node
            for _ in range(num_layers)
        ]


    def set_masks(self, keep_prob=0.7):
        for layer in self.layers:
            layer.set_random_mask(keep_prob)
            
            
    def forward(self, 
                x):
        activations = []
        out = x
        batch_size = x.shape[0]
        for idx, layer in enumerate(self.layers):
            out = layer(out)
            # Generate per-node noise for the whole batch
            noises = torch.stack([
                self.noise_funcs[idx][j](batch_size)  # shape (batch,)
                for j in range(out.shape[1])
            ], dim=1)  # shape (batch, nodes)
            out = out + noises
            out = self.activations[idx](out)
            activations.append(out)
        return torch.cat(activations, dim=1)


    def forward_with_weight_masks(self,
                                x,
                                noise_std=0.1, 
                                weight_masks=None):
        """
        x: (batch, in_features)
        noise_scales: list of (batch, layer_size) tensors or None
        weight_masks: list of (batch, layer_size, in_features) tensors or None
        """
        activations = []
        out = x
        batch_size = x.shape[0]
        for idx, layer in enumerate(self.layers):
            mask = weight_masks[idx] if weight_masks is not None else None
            out = layer(out, weight_mask=mask) if mask is not None else layer(out)
            # Generate per-node noise for the whole batch
            noises = torch.stack([
                self.noise_funcs[idx][j](batch_size)  # shape (batch,)
                for j in range(out.shape[1])
            ], dim=1)  # shape (batch, nodes)
            out = out + noises
            out = self.activations[idx](out)
            activations.append(out)
        return torch.cat(activations, dim=1)
        
    
    
    def forward_with_noise_scales(self,
                                x,
                                noise_scales=None,
                                return_noises=False):
        activations = []
        out = x
        batch_size = x.shape[0]
        all_noises = []
        all_scales = []
        for idx, layer in enumerate(self.layers):
            out = layer(out)
            noises = torch.stack([
                self.noise_funcs[idx][j](batch_size)
                for j in range(out.shape[1])
            ], dim=1)  # (batch, nodes)
            scales = noise_scales[idx] if noise_scales is not None else torch.ones_like(noises)
            noises = noises * scales
            out = out + noises
            out = self.activations[idx](out)
            activations.append(out)
            if return_noises:
                all_noises.append(noises)
                all_scales.append(scales)
        if return_noises:
            return torch.cat(activations, dim=1), all_noises, all_scales
        else:
            return torch.cat(activations, dim=1)

    
class StructuralCausalModel:
    def __init__(self,
                num_features: int = 3,
                min_num_layer: int = 3,
                max_num_layer: int = 5,
                min_hidden_size: int = 8,
                max_hidden_size: int = 8,
                device = 'cpu',
                outlier_type = 'contextual',
                drop_weight_prob: float = 0.7,
                ):
        self.device = device
        self.l, self.h = sample_layers_and_nodes(min_num_layer,max_num_layer,min_hidden_size, max_hidden_size)
        while self.l * self.h < num_features:
            self.l, self.h = sample_layers_and_nodes(min_num_layer,max_num_layer,min_hidden_size, max_hidden_size)
        self.activations = [sample_activation(device)[1] for _ in range(self.l)]
        self.mlp = SCM_MLP(self.h, self.l, activations=self.activations, device=device)
        self.mlp = self.mlp.to(device) 
        self.mlp.set_masks(keep_prob=drop_weight_prob) 
        self.num_features = num_features
        self.chosen_nodes = np.random.choice(self.l * self.h, self.num_features, replace=False)
        self.outlier_type = outlier_type

    @torch.no_grad()
    def sample_inliers(self, num_samples):
        # Generate random input (assume standard normal)
        x = torch.ones((num_samples, self.h), device=self.device)
        acts = self.mlp(x)  # shape: (num_samples, total_nodes)
        # Return only the selected nodes for each sample
        return acts[:, self.chosen_nodes]

    @torch.no_grad()
    def sample_prob_outliers(self, 
                            num_samples, 
                            high_noise=5.0, 
                            batch_factor=2):
        #let high prob be a function of num_features
        if self.num_features < 50:
            high_noise_prob = 0.2
        else:
            max_prob = 0.2
            min_prob = 0.01
            decay_scale = 50.0  # larger -> slower decay
            high_noise_prob = max(min_prob, max_prob * decay_scale / (self.num_features + decay_scale))
        
        collected = []
        while len(collected) < num_samples:
            batch_size = max(int((num_samples - len(collected)) * batch_factor), 10000)
            x = torch.ones((batch_size, self.h), device=self.device)
            layer_sizes = [layer.out_features for layer in self.mlp.layers]
            noise_scales = random_noise_scales_per_sample(
                batch_size, layer_sizes, 
                high_noise=high_noise, 
                chosen_nodes = self.chosen_nodes,
                high_noise_prob=high_noise_prob, 
                device=self.device
            )
            activations, all_noises, all_noise_scales = self.mlp.forward_with_noise_scales(
                x, noise_scales=noise_scales, return_noises=True
            )
            batch_mask = torch.ones(batch_size, dtype=torch.bool, device=x.device)
            for idx, (noises, scales) in enumerate(zip(all_noises, all_noise_scales)):
                high_noise_mask = (scales == high_noise)
                if high_noise_mask.any():
                    # For each node in this layer, get its mean and std
                    means = torch.tensor(
                        [float(getattr(self.mlp.noise_funcs[idx][j], 'mu', 0.0)) for j in range(noises.shape[1])],
                        device=x.device
                    )
                    stds = torch.tensor(
                        [float(getattr(self.mlp.noise_funcs[idx][j], 'sigma', 1.0)) for j in range(noises.shape[1])],
                        device=x.device
                    )
                    thresholds = means +  stds  # shape: (nodes,)
                    # Broadcast to batch shape
                    thresholds = thresholds.unsqueeze(0).expand_as(noises)
                    means = means.unsqueeze(0).expand_as(noises)
                    # Check (for high noise) if |noise - mean| >= threshold
                    valid = ((noises - means).abs() >= thresholds) | (~high_noise_mask)
                    valid = valid.all(dim=1)
                    batch_mask &= valid
            valid_idx = batch_mask.nonzero(as_tuple=True)[0]
            if len(valid_idx) > 0:
                acts_valid = activations[valid_idx][:, self.chosen_nodes]
                collected.append(acts_valid)
            total = sum(x.shape[0] for x in collected)
            if total >= num_samples:
                collected = torch.cat(collected)[:num_samples]
                break
        return collected
    
    @torch.no_grad()
    def sample_contextual_outliers(self, num_samples):
        x = torch.ones((num_samples, self.h), device=self.device)
        #make perturb prob decrease with number of features increase
        max_prob = 0.2
        min_prob = 0.03
        decay_scale = 50.0  # larger -> slower decay
        perturb_prob = max(min_prob, max_prob * decay_scale / (self.num_features + decay_scale))
        
        weight_masks = create_weight_mask(
            num_samples, 
            self.mlp.layers,
            chosen_nodes = self.chosen_nodes, 
            perturb_prob=perturb_prob,
            device=self.device
        )
        #print('draw weights', time.time()-start)
        acts = self.mlp.forward_with_weight_masks(x, weight_masks=weight_masks)
        return acts[:, self.chosen_nodes]


    @torch.no_grad()
    def draw_batched_data(self, 
                          num_inliers, 
                          num_local_anomalies):
        raw_inliers = self.sample_inliers(num_inliers)
        if self.outlier_type == 'prob':
            raw_local_anomalies = self.sample_prob_outliers(num_samples=num_local_anomalies)
        elif self.outlier_type == 'contextual':
            raw_local_anomalies = self.sample_contextual_outliers(num_samples=num_local_anomalies)
        return raw_inliers, raw_local_anomalies
    
    
    
def make_probSCM(max_feature_dim: int,
                 min_num_layer: int,
                 max_num_layer: int,
                 min_hidden_size: int,
                 max_hidden_size: int,
                 alpha: float,
                 beta: float,
                 device):
    return StructuralCausalModel(num_features = max_feature_dim,
                                 min_num_layer=min_num_layer,
                                 max_num_layer = max_num_layer,
                                 min_hidden_size = min_hidden_size,
                                 max_hidden_size = max_hidden_size,
                                 device = device,
                                 outlier_type = 'prob',
                                 drop_weight_prob = 0.6)


def make_contextualSCM(max_feature_dim: int,
                 min_num_layer: int,
                 max_num_layer: int,
                 min_hidden_size: int,
                 max_hidden_size: int,
                 alpha: float,
                 beta: float,
                 device):
    return StructuralCausalModel(num_features = max_feature_dim,
                                 min_num_layer=min_num_layer,
                                 max_num_layer = max_num_layer,
                                 min_hidden_size = min_hidden_size,
                                 max_hidden_size = max_hidden_size,
                                 device = device,
                                 outlier_type = 'contextual',
                                 drop_weight_prob = 0.6)


In [9]:
dimension_range = [(2,21),(21,41),(41,61),(61,81),(81,101)] #,(101,151),(151,251)]
base_seed= 52324 #52324 #1024 #52324

for j in range(len(dimension_range)):
    dimension = dimension_range[j]
    count = 0
    i = 0
    begin_idx = j * 32
    avg_roc = 0.0
    while count < 32:
        set_seed(base_seed + i)
        s = time.time()
        device = 'cuda:1'
        dim = np.random.randint(low=dimension[0], high=dimension[1])  # draw from [2, 20]
        
        #scm params
        max_num_layer = 5
        min_num_layer= max(int(np.sqrt(dim))-3,2)
        min_hidden_size = max(int(math.floor(dim / min_num_layer)) + 2 ,2)
        max_hidden_size = min(min_hidden_size+ 7,40)
        alpha = 1.5
        beta = 4.0
        
        num_inliers = np.random.randint(1_000, 5_000)
        outliers_ratio = np.random.uniform(0.05, 0.15)
        num_outliers = int(outliers_ratio * num_inliers)
        model = StructuralCausalModel(
            num_features=dim,
            min_num_layer=min_num_layer,
            max_num_layer=max_num_layer,
            min_hidden_size=min_hidden_size,
            max_hidden_size=max_hidden_size,
            device=device,
            drop_weight_prob=0.6,
            outlier_type='contextual'
        )

        # test the batch data follows the inliner/local_anomaly criterion
        X_in, X_out = model.draw_batched_data(num_inliers, num_outliers)
        #print(X_in.shape, X_out.shape)
        X   = torch.cat([X_in, X_out]).cpu().numpy()
        y   = np.concatenate([np.zeros(num_inliers), np.ones(num_outliers)])  # 1 ⇒ outlier

        contamination = num_outliers / (num_inliers + num_outliers)
        lof = LocalOutlierFactor(n_neighbors=20,
                                        contamination=contamination)
        _   = lof.fit_predict(X)                  
        scores = -lof.negative_outlier_factor_   
        auc = roc_auc_score(y, scores)
        #print(auc)

        if auc > 0.5 and auc < 0.95:
            saved = begin_idx + count
            #print('num_cluster', num_cluster, 'dim', dim, 'saved idx', saved, 'auc', auc)
            np.savez(f"/home/xding2/FoMo-0D-explore/data/synthetic_prior/test_data/scm_contextual_new/scmcontextualnew_{saved}.npz", X=X, y=y)
            count += 1
            avg_roc += auc
        i += 1
        #print(i)
        #print(count)

    print(f"For dimension {dimension[0]}, {dimension[1]}. Total rejected ratio: ", (count / i ), " avg auc: ", (avg_roc / count))

For dimension 2, 21. Total rejected ratio:  0.5714285714285714  avg auc:  0.8186352371523058
For dimension 21, 41. Total rejected ratio:  0.4383561643835616  avg auc:  0.8825727116126713
For dimension 41, 61. Total rejected ratio:  0.256  avg auc:  0.8845952385848704
For dimension 61, 81. Total rejected ratio:  0.07111111111111111  avg auc:  0.9225091042352097
For dimension 81, 101. Total rejected ratio:  0.03571428571428571  avg auc:  0.9302129366681782


In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score
import time
import math
import random

def lognormal_discrete(mu,
                       sigma,
                       minval:int,
                       maxval:int):
    # sample from lognormal distribtuion, making it discrete
    # input: mu, sigma, minval (int), maxval(int)
    # return: a integer value
    val = int(np.round(np.random.lognormal(mu, sigma)))
    return int(np.clip(val, minval, maxval))


def sample_layers_and_nodes(min_num_layer =2,
                            max_num_layer =5,
                            min_hidden_size = 3,
                            max_hidden_size = 8):
    #return: a randomly sampled hidden layer and number of layers
    l = lognormal_discrete(mu=0.7, sigma=0.4, minval=min_num_layer, maxval=max_num_layer)  # num layers
    h = lognormal_discrete(mu=1.2, sigma=0.5, minval=min_hidden_size, maxval=max_hidden_size)  # hidden size
    return l, h

@torch.no_grad()
def sample_noise_distribution(device='cpu'):
    mu = (torch.rand(1, device=device) - 0.5).item()
    sigma = (torch.rand(1, device=device) * (0.5 - 0.05) + 0.05).item()
    def noise_func(n):
        return torch.exp(mu + sigma * torch.randn(n, device=device))
    noise_func.mu = mu
    noise_func.sigma = sigma
    return noise_func

@torch.no_grad()
def sample_activation(device='cpu'):
    # sample activation functions (PyTorch version)
    activations = [
        ("tanh", torch.tanh),
        ("leaky_relu", lambda x: torch.where(x > 0, x, 0.01 * x)),
        ("elu", lambda x: torch.where(x > 0, x, torch.exp(x) - 1)),
        ("identity", lambda x: x),
    ]
    idx = torch.randint(0, len(activations), (1,), device=device).item()
    return activations[idx]


@torch.no_grad()
def random_noise_scales_per_sample(num_samples, layer_sizes, high_noise=5.0, high_noise_prob=0.2, device='cpu'):
    """
    Generate random noise scales for each sample and node, for each layer.
    Returns: List of tensors, each shape (num_samples, layer_size)
    """
    noise_scales = [
        torch.where(
            torch.rand(num_samples, n, device=device) < high_noise_prob,
            torch.full((num_samples, n), high_noise, device=device),
            torch.ones(num_samples, n, device=device)
        )
        for n in layer_sizes
    ]
    return noise_scales


@torch.no_grad()
def create_weight_mask(
    num_samples, layers, chosen_nodes, perturb_prob=0.5, device='cpu'
):
    """
    Vectorized version: No per-sample loop.
    For each sample and layer, ensures that at least one parent of a chosen node is perturbed.
    """
    masks = []
    node_layer_size = layers[0].weight.shape[0]  # assumes all layers same size

    for l, layer in enumerate(layers):
        weight = layer.weight  # (out_features, in_features)
        perturbable = (torch.abs(weight) > 0.5)  # (out, in)
        mask = torch.ones(num_samples, *weight.shape, device=device)

        perturb_mask = (torch.rand(num_samples, *weight.shape, device=device) < perturb_prob) & perturbable.unsqueeze(0)
        flip_mask = torch.rand(num_samples, *weight.shape, device=device) < 0.5

        mask[perturb_mask & flip_mask] = -1.0
        mask[perturb_mask & (~flip_mask)] = 0.0

        # Find chosen nodes for this layer (global to local index)
        chosen_nodes_this_layer = [idx for idx in chosen_nodes if (idx // node_layer_size) == l]
        if len(chosen_nodes_this_layer) == 0:
            masks.append(mask)
            continue

        for cidx in chosen_nodes_this_layer:
            node_idx = cidx % node_layer_size  # local node index for this layer

            # For all samples: find if any parent is perturbed (and perturbable) for this node
            perturbed = ((mask[:, node_idx, :] != 1.0) & perturbable[node_idx, :])  # (num_samples, in_features)
            any_perturbed = perturbed.any(dim=1)  # (num_samples,)

            need_perturb = (~any_perturbed)  # (num_samples,)
            num_to_fix = need_perturb.sum().item()
            if num_to_fix == 0:
                continue

            # For those samples, pick a random eligible parent and force a perturbation
            eligible_parents = perturbable[node_idx, :].nonzero(as_tuple=True)[0]
            if len(eligible_parents) == 0:
                continue

            # Pick a random parent for each sample needing fix
            rand_idx = torch.randint(0, len(eligible_parents), (num_to_fix,), device=device)
            parent_idx = eligible_parents[rand_idx]  # (num_to_fix,)
            sample_idx = need_perturb.nonzero(as_tuple=True)[0]  # (num_to_fix,)

            # Randomly decide flip or zero
            random_flip = (torch.rand(num_to_fix, device=device) < 0.5)
            mask[sample_idx, node_idx, parent_idx] = torch.where(random_flip, -1.0, 0.0)

        masks.append(mask)
    return masks



class MaskedLinear(nn.Linear):
    def __init__(self, in_features, out_features, min_abs=0.35, device='cpu'):
        super().__init__(in_features, out_features, False)
        # Sample weights from N(0, 1)
        with torch.no_grad():
            w = torch.normal(mean=0., std=1., size=self.weight.shape, device=device)
            # abs_w = torch.clamp(torch.abs(w), min=torch.tensor(min_abs,device=device))
            # w_clipped = torch.sign(w) * abs_w
            self.weight.copy_(w) #_clipped)
        self.mask = nn.Parameter(torch.ones_like(self.weight), requires_grad=False)


    def set_random_mask(self, keep_prob=0.7):
        with torch.no_grad():
            self.mask[:] = (torch.rand_like(self.mask) < keep_prob).float()

    def forward(self, input, weight_mask=None):
        # input: (batch, in_features)
        # self.weight: (out_features, in_features)
        # weight_mask: (batch, out_features, in_features) or None
        masked_weight = self.weight * self.mask
        if weight_mask is None:
            return nn.functional.linear(input, masked_weight, None)
        # Use per-sample masked weights (batched matmul)
        # weight_mask shape: (batch, out_features, in_features)
        # input shape: (batch, in_features)
        # Expand masked_weight for batch: (1, out_features, in_features)
        batch = input.size(0)
        masked_weight = masked_weight.unsqueeze(0)  # (1, out_features, in_features)
        # Broadcast for batch
        weight = masked_weight.expand(batch, -1, -1) * weight_mask  # (batch, out_features, in_features)
        # Batched matmul: input (batch, in_features) × weight.transpose(-2, -1) (batch, in_features, out_features)
        # Result: (batch, out_features)
        out = torch.bmm(input.unsqueeze(1), weight.transpose(1,2)).squeeze(1)
        return out




class SCM_MLP(nn.Module):
    def __init__(self, hidden_dim, num_layers, activations,device='cuda'):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(MaskedLinear(hidden_dim, hidden_dim,device=device))
        assert len(activations) == len(self.layers)
        self.activations = activations
        
        # Per-node noise distributions for each layer and neuron
        self.noise_funcs = [
            [sample_noise_distribution(device) for _ in range(hidden_dim)]  # per node
            for _ in range(num_layers)
        ]


    def set_masks(self, keep_prob=0.7):
        for layer in self.layers:
            layer.set_random_mask(keep_prob)
            
            
    def forward(self, 
                x):
        activations = []
        out = x
        batch_size = x.shape[0]
        for idx, layer in enumerate(self.layers):
            out = layer(out)
            # Generate per-node noise for the whole batch
            noises = torch.stack([
                self.noise_funcs[idx][j](batch_size)  # shape (batch,)
                for j in range(out.shape[1])
            ], dim=1)  # shape (batch, nodes)
            out = out + noises
            out = self.activations[idx](out)
            activations.append(out)
        return torch.cat(activations, dim=1)


    def forward_with_weight_masks(self,
                                x,
                                noise_std=0.1, 
                                weight_masks=None):
        """
        x: (batch, in_features)
        noise_scales: list of (batch, layer_size) tensors or None
        weight_masks: list of (batch, layer_size, in_features) tensors or None
        """
        activations = []
        out = x
        batch_size = x.shape[0]
        for idx, layer in enumerate(self.layers):
            mask = weight_masks[idx] if weight_masks is not None else None
            out = layer(out, weight_mask=mask) if mask is not None else layer(out)
            # Generate per-node noise for the whole batch
            noises = torch.stack([
                self.noise_funcs[idx][j](batch_size)  # shape (batch,)
                for j in range(out.shape[1])
            ], dim=1)  # shape (batch, nodes)
            out = out + noises
            out = self.activations[idx](out)
            activations.append(out)
        return torch.cat(activations, dim=1)
        
    
    
    def forward_with_noise_scales(self,
                                x,
                                noise_scales=None,
                                return_noises=False):
        activations = []
        out = x
        batch_size = x.shape[0]
        all_noises = []
        all_scales = []
        for idx, layer in enumerate(self.layers):
            out = layer(out)
            noises = torch.stack([
                self.noise_funcs[idx][j](batch_size)
                for j in range(out.shape[1])
            ], dim=1)  # (batch, nodes)
            scales = noise_scales[idx] if noise_scales is not None else torch.ones_like(noises)
            noises = noises * scales
            out = out + noises
            out = self.activations[idx](out)
            activations.append(out)
            if return_noises:
                all_noises.append(noises)
                all_scales.append(scales)
        if return_noises:
            return torch.cat(activations, dim=1), all_noises, all_scales
        else:
            return torch.cat(activations, dim=1)

    
class StructuralCausalModel:
    def __init__(self,
                num_features: int = 3,
                min_num_layer: int = 3,
                max_num_layer: int = 5,
                min_hidden_size: int = 8,
                max_hidden_size: int = 8,
                device = 'cpu',
                outlier_type = 'contextual',
                drop_weight_prob: float = 0.7,
                ):
        self.device = device
        self.l, self.h = sample_layers_and_nodes(min_num_layer,max_num_layer,min_hidden_size, max_hidden_size)
        while self.l * self.h < num_features:
            self.l, self.h = sample_layers_and_nodes(min_num_layer,max_num_layer,min_hidden_size, max_hidden_size)
        self.activations = [sample_activation(device)[1] for _ in range(self.l)]
        self.mlp = SCM_MLP(self.h, self.l, activations=self.activations, device=device)
        self.mlp = self.mlp.to(device) 
        self.mlp.set_masks(keep_prob=drop_weight_prob) 
        self.num_features = num_features
        self.chosen_nodes = np.random.choice(self.l * self.h, self.num_features, replace=False)
        self.outlier_type = outlier_type

    @torch.no_grad()
    def sample_inliers(self, num_samples):
        # Generate random input (assume standard normal)
        x = torch.ones((num_samples, self.h), device=self.device)
        acts = self.mlp(x)  # shape: (num_samples, total_nodes)
        # Return only the selected nodes for each sample
        return acts[:, self.chosen_nodes]

    @torch.no_grad()
    def sample_prob_outliers(self, 
                            num_samples, 
                            high_noise=5.0, 
                            high_noise_prob=0.2, 
                            batch_factor=2):
        collected = []
        while len(collected) < num_samples:
            batch_size = max(int((num_samples - len(collected)) * batch_factor), 10000)
            x = torch.ones((batch_size, self.h), device=self.device)
            layer_sizes = [layer.out_features for layer in self.mlp.layers]
            noise_scales = random_noise_scales_per_sample(
                batch_size, layer_sizes, 
                high_noise=high_noise, 
                high_noise_prob=high_noise_prob, 
                device=self.device
            )
            activations, all_noises, all_noise_scales = self.mlp.forward_with_noise_scales(
                x, noise_scales=noise_scales, return_noises=True
            )
            batch_mask = torch.ones(batch_size, dtype=torch.bool, device=x.device)
            for idx, (noises, scales) in enumerate(zip(all_noises, all_noise_scales)):
                high_noise_mask = (scales == high_noise)
                if high_noise_mask.any():
                    # For each node in this layer, get its mean and std
                    means = torch.tensor(
                        [float(getattr(self.mlp.noise_funcs[idx][j], 'mu', 0.0)) for j in range(noises.shape[1])],
                        device=x.device
                    )
                    stds = torch.tensor(
                        [float(getattr(self.mlp.noise_funcs[idx][j], 'sigma', 1.0)) for j in range(noises.shape[1])],
                        device=x.device
                    )
                    thresholds = means +  stds  # shape: (nodes,)
                    # Broadcast to batch shape
                    thresholds = thresholds.unsqueeze(0).expand_as(noises)
                    means = means.unsqueeze(0).expand_as(noises)
                    # Check (for high noise) if |noise - mean| >= threshold
                    valid = ((noises - means).abs() >= thresholds) | (~high_noise_mask)
                    valid = valid.all(dim=1)
                    batch_mask &= valid
            valid_idx = batch_mask.nonzero(as_tuple=True)[0]
            if len(valid_idx) > 0:
                acts_valid = activations[valid_idx][:, self.chosen_nodes]
                collected.append(acts_valid)
            total = sum(x.shape[0] for x in collected)
            if total >= num_samples:
                collected = torch.cat(collected)[:num_samples]
                break
        return collected

    # @torch.no_grad()
    # def sample_contextual_outliers(self, num_samples):
    #     x = torch.ones((num_samples, self.h), device=self.device)
    #     weight_masks = create_weight_mask(
    #         num_samples, self.l, self.h, self.h, self.chosen_nodes, device=self.device,perturb_prob=0.2
    #     )
    #     print(weight_masks)
    #     print(self.chosen_nodes)
    #     acts = self.mlp.forward_with_weight_masks(x, weight_masks=weight_masks)
    #     return acts[:, self.chosen_nodes]
    
    
    @torch.no_grad()
    def sample_contextual_outliers(self, num_samples, perturb_prob=0.2):
        x = torch.ones((num_samples, self.h), device=self.device)
        #start = time.time()
        weight_masks = create_weight_mask(
            num_samples, self.mlp.layers, chosen_nodes = self.chosen_nodes, perturb_prob=perturb_prob, device=self.device
        )
        #print('draw weights', time.time()-start)
        acts = self.mlp.forward_with_weight_masks(x, weight_masks=weight_masks)
        return acts[:, self.chosen_nodes]


    @torch.no_grad()
    def draw_batched_data(self, 
                          num_inliers, 
                          num_local_anomalies):
        #start = time.time()
        raw_inliers = self.sample_inliers(num_inliers)
        #print('sample inliers', time.time()-start)
        if self.outlier_type == 'prob':
            raw_local_anomalies = self.sample_prob_outliers(num_samples=num_local_anomalies)
        elif self.outlier_type == 'contextual':
            raw_local_anomalies = self.sample_contextual_outliers(num_samples=num_local_anomalies)
        return raw_inliers, raw_local_anomalies
    
    
    
def make_probSCM(max_feature_dim: int,
                 min_num_layer: int,
                 max_num_layer: int,
                 min_hidden_size: int,
                 max_hidden_size: int,
                 alpha: float,
                 beta: float,
                 device):
    return StructuralCausalModel(num_features = max_feature_dim,
                                 min_num_layer=min_num_layer,
                                 max_num_layer = max_num_layer,
                                 min_hidden_size = min_hidden_size,
                                 max_hidden_size = max_hidden_size,
                                 device = device,
                                 outlier_type = 'prob',
                                 drop_weight_prob = 0.6)


def make_contextualSCM(max_feature_dim: int,
                 min_num_layer: int,
                 max_num_layer: int,
                 min_hidden_size: int,
                 max_hidden_size: int,
                 alpha: float,
                 beta: float,
                 device):
    return StructuralCausalModel(num_features = max_feature_dim,
                                 min_num_layer=min_num_layer,
                                 max_num_layer = max_num_layer,
                                 min_hidden_size = min_hidden_size,
                                 max_hidden_size = max_hidden_size,
                                 device = device,
                                 outlier_type = 'contextual',
                                 drop_weight_prob = 0.6)
    


In [ ]:
dimension_range = [(2,21),(21,41),(41,61),(61,81),(81,101),(101,151),(151,251)]
base_seed= 52324 #52324 #1024 #52324

for j in range(len(dimension_range)):
    dimension = dimension_range[j]
    count = 0
    i = 0
    begin_idx = j * 32
    avg_roc = 0.0
    while count < 32:
        set_seed(base_seed + i)
        s = time.time()
        device = 'cuda:1'
        dim = np.random.randint(low=dimension[0], high=dimension[1])  # draw from [2, 20]
        
        #scm params
        max_num_layer = 5
        min_num_layer= max(int(np.sqrt(dim))-3,2)
        min_hidden_size = max(int(math.floor(dim / min_num_layer)) + 2 ,2)
        max_hidden_size = min(min_hidden_size+ 7,40)
        alpha = 1.5
        beta = 4.0
        
        num_inliers = np.random.randint(1_000, 5_000)
        outliers_ratio = np.random.uniform(0.05, 0.15)
        num_outliers = int(outliers_ratio * num_inliers)
        model = StructuralCausalModel(
            num_features=dim,
            min_num_layer=min_num_layer,
            max_num_layer=max_num_layer,
            min_hidden_size=min_hidden_size,
            max_hidden_size=max_hidden_size,
            device=device,
            drop_weight_prob=0.6,
            outlier_type='prob'
        )

        # test the batch data follows the inliner/local_anomaly criterion
        X_in, X_out = model.draw_batched_data(num_inliers, num_outliers)
        #print(X_in.shape, X_out.shape)
        X   = torch.cat([X_in, X_out]).cpu().numpy()
        y   = np.concatenate([np.zeros(num_inliers), np.ones(num_outliers)])  # 1 ⇒ outlier

        contamination = num_outliers / (num_inliers + num_outliers)
        lof = LocalOutlierFactor(n_neighbors=20,
                                        contamination=contamination)
        _   = lof.fit_predict(X)                  
        scores = -lof.negative_outlier_factor_   
        auc = roc_auc_score(y, scores)
        #print(auc)

        if auc > 0.5 and auc < 0.95:
            saved = begin_idx + count
            #print('num_cluster', num_cluster, 'dim', dim, 'saved idx', saved, 'auc', auc)
            #np.savez(f"synthetic_prior/test_data/scm_contextual/scmcontextual_{saved}.npz", X=X, y=y)
            count += 1
            avg_roc += auc
        i += 1
        #print(i)
        #print(count)

    print(f"For dimension {dimension[0]}, {dimension[1]}. Total rejected ratio: ", (count / i ), " avg auc: ", (avg_roc / count))

For dimension 2, 21. Total rejected ratio:  0.9411764705882353  avg auc:  0.8294377122733139
For dimension 21, 41. Total rejected ratio:  0.42105263157894735  avg auc:  0.8955707369491731
For dimension 41, 61. Total rejected ratio:  0.43243243243243246  avg auc:  0.9065543992538375
For dimension 61, 81. Total rejected ratio:  0.43243243243243246  avg auc:  0.8857821394269585
For dimension 81, 101. Total rejected ratio:  0.3516483516483517  avg auc:  0.9059429635874696
